# 05 - Grad-CAM temporal explainability evaluation

This notebook loads trained 2D U-Net and ConvLSTM U-Net checkpoints and computes Grad-CAM-style temporal explanations on the official EchoNet-Dynamic test split only. It does not train models.

In [ ]:
# Kaggle setup. Skip this cell when the environment already satisfies requirements.txt.
%pip install -q monai opencv-python-headless pandas matplotlib tqdm

In [ ]:
from pathlib import Path
import json
import os
import random
import sys
import warnings

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.dataset import split_by_echonet_filelist
from src.gradcam import convlstm_gradcam, find_last_conv2d_name, unet_framewise_gradcam
from src.model import build_unet
from src.temporal_dataset_variable_stride import (
    EchoNetTemporalVariableStrideDataset,
    build_fps_lookup,
    load_temporal_metadata,
)
from src.temporal_evaluation import (
    aggregate_metrics,
    compute_temporal_saliency_metrics,
)
from src.temporal_model import build_convlstm_unet
from src.utils import load_echonet_tables, set_seed
from src.visualization import sanitize_name, save_heatmaps, save_metric_plots, save_overlay_grid

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## Configuration

In [ ]:
RUN_MODE = 'smoke'  # use 'full' for the entire official test set
SMOKE_RANDOM_SEED = 42
SALIENCY_PERCENTILE = 80.0
MAX_OVERLAY_SAMPLES = 1 if RUN_MODE == 'smoke' else 10

RAW_DIR = Path(os.environ.get('ECHONET_RAW_DIR', PROJECT_ROOT / 'data' / 'raw' / 'EchoNet-Dynamic'))
PROCESSED_DIR = Path(os.environ.get('ECHONET_PROCESSED_DIR', PROJECT_ROOT / 'data' / 'processed'))
VIDEOS_DIR = RAW_DIR / 'Videos'
OUTPUT_DIR = Path('/kaggle/working/outputs/runs/gradcam_temporal_evaluation') if Path('/kaggle/working').exists() else PROJECT_ROOT / 'outputs' / 'runs' / 'gradcam_temporal_evaluation'
PSEUDOLABEL_RUN_DIR = Path(os.environ.get('PSEUDOLABEL_RUN_DIR', '/kaggle/input/segmentation-pseudolabels/segmentation_pseudolabels' if Path('/kaggle/input').exists() else PROJECT_ROOT / 'outputs' / 'runs' / 'segmentation_pseudolabels'))

HEATMAP_DIR = OUTPUT_DIR / 'heatmaps'  # legacy path; new reusable arrays are saved under CAM_DIR
CAM_DIR = OUTPUT_DIR / 'cams'
OVERLAY_DIR = OUTPUT_DIR / 'overlays'
QUAL_DIR = OUTPUT_DIR / 'visualizations' / 'model_layer_comparisons'
METRICS_DIR = OUTPUT_DIR / 'metrics'
FIGURES_DIR = OUTPUT_DIR / 'figures'
TABLES_DIR = OUTPUT_DIR / 'tables'
for directory in [HEATMAP_DIR, CAM_DIR, OVERLAY_DIR, QUAL_DIR, METRICS_DIR, FIGURES_DIR, TABLES_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

CONVLSTM_TARGET_LAYERS = ['bottleneck_encoder', 'temporal_bottleneck', 'decoder3']
CONVLSTM_STRIDES = [1, 4, 6, 8, 10]
SEQUENCE_LENGTH = 5
IMAGE_SIZE = (112, 112)
BATCH_SIZE = 1
NUM_WORKERS = 0

print(f'Raw data: {RAW_DIR}')
print(f'Processed data: {PROCESSED_DIR}')
print(f'Output directory: {OUTPUT_DIR}')
print(f'Pseudolabel run directory: {PSEUDOLABEL_RUN_DIR}')
print(f'Max unique samples with saved overlays: {MAX_OVERLAY_SAMPLES}')

## Checkpoint discovery

In [ ]:
def first_existing(paths):
    for path in paths:
        path = Path(path)
        if path.exists():
            return path
    return None


VARIABLE_STRIDE_ROOT = first_existing(
    [
        Path(os.environ.get('CONVLSTM_VARIABLE_STRIDE_RUN_DIR', '')),
        PROJECT_ROOT / 'outputs' / 'runs' / 'convlstm_unet_variable_strides_06_21',
        PROJECT_ROOT / 'outputs' / 'runs' / 'convlstm_unet_variable_strides',
    ]
)
STRIDE1_ROOT = first_existing(
    [
        Path(os.environ.get('CONVLSTM_STRIDE1_RUN_DIR', '')),
        PROJECT_ROOT / 'outputs' / 'runs' / 'convlstm_unet',
        PROJECT_ROOT / 'outputs' / 'runs' / 'ConvLSTM_Unet_06_11',
    ]
)
UNET_CHECKPOINT_PATH = first_existing(
    [
        Path(os.environ.get('UNET_CHECKPOINT_PATH', '')),
        PROJECT_ROOT / 'outputs' / 'checkpoints' / 'best_unet.pt',
        PROJECT_ROOT / 'outputs' / 'runs' / 'unet' / 'checkpoints' / 'best_unet.pt',
    ]
)

def convlstm_checkpoint_for_stride(stride: int):
    if stride == 1 and STRIDE1_ROOT is not None:
        return STRIDE1_ROOT / 'checkpoints' / 'best_model.pt'
    if stride != 1 and VARIABLE_STRIDE_ROOT is not None:
        return VARIABLE_STRIDE_ROOT / f'convlstm_unet_stride_{stride}' / 'checkpoints' / 'best_model.pt'
    return None


def config_for_stride(stride: int):
    if stride == 1 and STRIDE1_ROOT is not None:
        config_path = STRIDE1_ROOT / 'config.json'
    elif VARIABLE_STRIDE_ROOT is not None:
        config_path = VARIABLE_STRIDE_ROOT / f'convlstm_unet_stride_{stride}' / 'config.json'
    else:
        config_path = None
    if config_path and config_path.exists():
        with config_path.open('r', encoding='utf-8') as file:
            return json.load(file)
    return {'channels': [16, 32, 64, 128], 'sequence_length': SEQUENCE_LENGTH, 'image_size': list(IMAGE_SIZE)}


checkpoint_rows = []
for stride in CONVLSTM_STRIDES:
    path = convlstm_checkpoint_for_stride(stride)
    exists = bool(path and path.exists())
    checkpoint_rows.append({'model_family': 'convlstm_unet', 'stride': stride, 'checkpoint_path': str(path), 'exists': exists})
checkpoint_rows.append({'model_family': 'unet_2d', 'stride': None, 'checkpoint_path': str(UNET_CHECKPOINT_PATH), 'exists': bool(UNET_CHECKPOINT_PATH and UNET_CHECKPOINT_PATH.exists())})
checkpoint_df = pd.DataFrame(checkpoint_rows)
checkpoint_df.to_csv(TABLES_DIR / 'checkpoint_discovery.csv', index=False)
checkpoint_df

## Official test split only

In [ ]:
metadata_path = PROCESSED_DIR / 'metadata.csv'
assert metadata_path.exists(), 'metadata.csv is required from notebook 02 preprocessing.'
assert VIDEOS_DIR.exists(), f'Videos directory not found: {VIDEOS_DIR}'

samples = load_temporal_metadata(metadata_path)
file_list, _ = load_echonet_tables(RAW_DIR)
fps_by_video = build_fps_lookup(file_list)
_, _, test_samples = split_by_echonet_filelist(samples, file_list)
assert len(test_samples) > 0, 'Official EchoNet test split has no matched processed samples.'

if RUN_MODE == 'smoke':
    rng = random.Random(SMOKE_RANDOM_SEED)
    eval_samples = [rng.choice(test_samples)]
else:
    eval_samples = test_samples

print(f'Official test samples available: {len(test_samples):,}')
print(f'Evaluation samples selected for {RUN_MODE}: {len(eval_samples):,}')

## Model loading helpers

In [ ]:
def load_state_dict_from_checkpoint(model, checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    state_dict = checkpoint.get('model_state_dict', checkpoint)
    model.load_state_dict(state_dict)
    model.eval()
    return model


def build_convlstm_for_stride(stride: int):
    config = config_for_stride(stride)
    model = build_convlstm_unet(
        in_channels=1,
        out_channels=1,
        channels=tuple(config.get('channels', [16, 32, 64, 128])),
    ).to(device)
    return model, config


def build_unet_baseline():
    model = build_unet(
        spatial_dims=2,
        in_channels=1,
        out_channels=1,
        channels=(16, 32, 64, 128, 256),
        strides=(2, 2, 2, 2),
        num_res_units=2,
    ).to(device)
    return model


def find_unet_encoder_bottleneck_conv2d_name(model) -> str:
    """Select the deepest/highest-channel Conv2d, excluding the final 1-channel output."""
    candidates = []
    for name, module in model.named_modules():
        if isinstance(module, torch.nn.Conv2d) and int(module.out_channels) > 1:
            candidates.append((int(module.out_channels), name))
    if not candidates:
        raise ValueError('No non-output Conv2d layer found for U-Net bottleneck Grad-CAM.')
    max_channels = max(channels for channels, _name in candidates)
    return [name for channels, name in candidates if channels == max_channels][-1]


def make_temporal_loader(stride: int):
    dataset = EchoNetTemporalVariableStrideDataset(
        eval_samples,
        videos_dir=VIDEOS_DIR,
        sequence_length=SEQUENCE_LENGTH,
        temporal_stride=stride,
        image_size=IMAGE_SIZE,
        augment=False,
        fps_by_video=fps_by_video,
    )
    return DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)


def batch_item(batch, key, index=0, default=None):
    value = batch.get(key, default)
    if isinstance(value, torch.Tensor):
        selected = value[index]
        return selected.detach().cpu().tolist() if selected.ndim > 0 else selected.detach().cpu().item()
    if isinstance(value, (list, tuple)):
        return value[index]
    return value


class DensePseudolabelStore:
    """Load packed per-video masks generated by notebook 18."""

    def __init__(self, run_dir: Path, image_size: tuple[int, int]) -> None:
        self.run_dir = Path(run_dir)
        self.image_size = tuple(image_size)
        manifest_path = self.run_dir / 'manifests' / 'pseudolabel_frame_manifest.csv'
        if not manifest_path.exists():
            raise FileNotFoundError(
                f'Missing pseudolabel manifest: {manifest_path}. Run notebook 18 first, or set PSEUDOLABEL_RUN_DIR.'
            )
        self.frame_manifest = pd.read_csv(manifest_path)
        self.lookup = {}
        for row in self.frame_manifest.itertuples(index=False):
            raw_path = Path(str(row.mask_npz_path))
            if raw_path.exists():
                mask_path = raw_path
            else:
                # Notebook 18 may have written absolute /kaggle/working paths before
                # the outputs were uploaded as a Kaggle dataset. Remap by filename.
                remapped = self.run_dir / 'masks_by_video' / raw_path.name
                if remapped.exists():
                    mask_path = remapped
                else:
                    mask_path = raw_path
            self.lookup[(str(row.video_id), int(row.frame_idx))] = (
                mask_path,
                int(row.mask_row_index),
                bool(row.is_ground_truth),
            )
        self._cache: dict[Path, dict[str, np.ndarray]] = {}

    def _load_npz(self, path: Path) -> dict[str, np.ndarray]:
        if path not in self._cache:
            with np.load(path, allow_pickle=False) as data:
                shape = tuple(data['mask_shape'].astype(int).tolist())
                packed = data['masks_packed']
                masks = np.unpackbits(packed, axis=-1)[..., : shape[-1]].reshape(shape).astype(bool)
                self._cache[path] = {
                    'masks': masks,
                    'frame_indices': data['frame_indices'].astype(np.int32),
                    'is_ground_truth': data['is_ground_truth'].astype(bool),
                }
        return self._cache[path]

    def get(self, video_id: str, frame_idx: int) -> tuple[np.ndarray, bool]:
        key = (str(video_id), int(frame_idx))
        if key not in self.lookup:
            raise KeyError(f'Missing dense pseudolabel for {video_id} frame {frame_idx}.')
        path, row_idx, is_gt = self.lookup[key]
        payload = self._load_npz(path)
        return payload['masks'][row_idx].astype(np.float32), bool(is_gt)

    def stack_for_frames(self, video_id: str, frame_indices: list[int]) -> tuple[np.ndarray, list[bool]]:
        masks = []
        is_gt = []
        for frame_idx in frame_indices:
            mask, gt = self.get(video_id, int(frame_idx))
            masks.append(mask)
            is_gt.append(gt)
        return np.stack(masks, axis=0).astype(np.float32), is_gt


pseudolabel_store = DensePseudolabelStore(PSEUDOLABEL_RUN_DIR, IMAGE_SIZE)
print(f'Loaded dense pseudolabel manifest rows: {len(pseudolabel_store.frame_manifest):,}')


def video_path(video_id: str) -> Path:
    filename = video_id if str(video_id).lower().endswith('.avi') else f'{video_id}.avi'
    return VIDEOS_DIR / filename


def read_temporal_sequence(video_id: str, center_frame_idx: int, stride: int) -> tuple[torch.Tensor, list[int]]:
    path = video_path(video_id)
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        raise FileNotFoundError(f'Could not open video: {path}')
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frames = []
    frame_indices = []
    radius = SEQUENCE_LENGTH // 2
    for rel in range(-radius, radius + 1):
        frame_idx = min(max(int(center_frame_idx) + rel * int(stride), 0), frame_count - 1)
        frame_indices.append(frame_idx)
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ok, frame_bgr = cap.read()
        if not ok or frame_bgr is None:
            cap.release()
            raise ValueError(f'Could not read frame {frame_idx} from {path}')
        gray = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)
        gray = cv2.resize(gray, (IMAGE_SIZE[1], IMAGE_SIZE[0]), interpolation=cv2.INTER_AREA)
        frames.append(gray.astype(np.float32) / 255.0)
    cap.release()
    sequence = torch.from_numpy(np.stack(frames, axis=0)).unsqueeze(1).unsqueeze(0).contiguous()
    return sequence, frame_indices


def save_compressed_cam_arrays(heatmaps: np.ndarray, output_path: Path, metadata: dict) -> Path:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    heatmaps = np.asarray(heatmaps, dtype=np.float32)
    frame_norm = np.zeros_like(heatmaps, dtype=np.float32)
    for idx, heatmap in enumerate(heatmaps):
        hm = heatmap - float(np.nanmin(heatmap))
        max_value = float(np.nanmax(hm))
        frame_norm[idx] = hm / max_value if max_value > 0 else hm
    clip = heatmaps - float(np.nanmin(heatmaps))
    clip_max = float(np.nanmax(clip))
    clip_norm = clip / clip_max if clip_max > 0 else clip
    np.savez_compressed(
        output_path,
        positive_clip_normalized_uint8=np.round(np.clip(clip_norm, 0, 1) * 255).astype(np.uint8),
        frame_normalized_uint8=np.round(np.clip(frame_norm, 0, 1) * 255).astype(np.uint8),
        metadata_json=np.asarray(json.dumps(metadata)),
    )
    return output_path


## Run Grad-CAM evaluation

In [ ]:
per_sample_rows = []
frame_overlap_rows = []
cam_manifest_rows = []
qualitative_records = []
overlay_sample_ids = set()
qualitative_by_layer: dict[str, dict[str, dict[str, object]]] = {}


def should_save_overlay(sample_id: str) -> bool:
    if sample_id in overlay_sample_ids:
        return True
    if len(overlay_sample_ids) < MAX_OVERLAY_SAMPLES:
        overlay_sample_ids.add(sample_id)
        return True
    return False


def cleanup_gradcam_gpu(model=None):
    if model is not None:
        model.zero_grad(set_to_none=True)
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def prediction_mask_and_scores(logits_np: np.ndarray, target_mask_np: np.ndarray, threshold: float = 0.5) -> tuple[np.ndarray, float, float]:
    logits = np.asarray(logits_np, dtype=np.float32).squeeze()
    probs = 1.0 / (1.0 + np.exp(-logits))
    pred_mask = probs >= threshold
    target_mask = np.asarray(target_mask_np).squeeze().astype(bool)
    intersection = float(np.logical_and(pred_mask, target_mask).sum())
    pred_sum = float(pred_mask.sum())
    target_sum = float(target_mask.sum())
    dice = (2.0 * intersection) / (pred_sum + target_sum + 1e-7)
    union = float(np.logical_or(pred_mask, target_mask).sum())
    iou = intersection / (union + 1e-7)
    return pred_mask.astype(np.float32), float(dice), float(iou)


def record_frame_overlaps(metric_row: dict, model_id: str, layer_name: str, frame_indices: list[int], mask_is_gt: list[bool]) -> None:
    for idx, frame_idx in enumerate(frame_indices):
        key = f'saliency_mask_overlap_frame_{idx}'
        frame_overlap_rows.append({
            'sample_id': metric_row['sample_id'],
            'video_id': metric_row['video_id'],
            'model_family': metric_row['model_family'],
            'model_id': model_id,
            'target_layer': layer_name,
            'temporal_stride': metric_row['temporal_stride'],
            'sequence_position': idx,
            'frame_idx': int(frame_idx),
            'saliency_mask_overlap': metric_row.get(key, np.nan),
            'mask_source': 'ground_truth' if bool(mask_is_gt[idx]) else 'pseudolabel',
        })


def qualitative_layer_group(layer_name: str, model_id: str) -> str:
    """Group comparable U-Net and ConvLSTM layers into the same qualitative figure."""
    del model_id
    if layer_name in {'convlstm_bottleneck_encoder', 'unet_encoder_bottleneck'}:
        return 'encoder_bottleneck_comparison'
    if layer_name in {'convlstm_decoder3_sliding_window', 'unet_final_convolution'}:
        return 'decoder3_unet_final_comparison'
    return sanitize_name(layer_name)


def maybe_store_qualitative(
    layer_name: str,
    sample_id: str,
    model_id: str,
    frames_np: np.ndarray,
    heatmaps: np.ndarray,
    dense_masks: np.ndarray,
    frame_indices: list[int],
    metric_row: dict,
    pred_mask: np.ndarray,
    dice: float,
    pred_iou: float,
) -> None:
    """Store full 5-frame sequences plus center prediction for comparison figures."""
    if not should_save_overlay(sample_id):
        return
    key = qualitative_layer_group(layer_name, model_id)
    sample_bucket = qualitative_by_layer.setdefault(key, {}).setdefault(sample_id, {})
    sample_bucket[model_id] = {
        'frames': np.asarray(frames_np, dtype=np.float32).copy(),
        'heatmaps': np.asarray(heatmaps, dtype=np.float32).copy(),
        'masks': np.asarray(dense_masks, dtype=np.float32).copy(),
        'pred_mask': np.asarray(pred_mask, dtype=np.float32).copy(),
        'dice': float(dice),
        'pred_iou': float(pred_iou),
        'frame_indices': [int(x) for x in frame_indices],
        'metric_row': dict(metric_row),
        'target_layer': layer_name,
    }

def write_cam_artifact(heatmaps: np.ndarray, model_id: str, layer_name: str, sample_id: str, metadata: dict) -> str:
    path = CAM_DIR / sanitize_name(layer_name) / model_id / f'{model_id}_{sanitize_name(layer_name)}_{sample_id}_cams.npz'
    if not path.exists():
        save_compressed_cam_arrays(heatmaps, path, metadata)
    return str(path)


def evaluate_convlstm_decoder3_sliding(model, stride: int, batch, checkpoint_path: Path, model_id: str):
    display_layer = 'convlstm_decoder3_sliding_window'
    sample_id = sanitize_name(batch_item(batch, 'id'))
    video_id = batch_item(batch, 'video_id')
    center_frame_idx = int(batch_item(batch, 'frame_idx'))
    original_frame_indices = [int(x) for x in batch_item(batch, 'frame_indices')]
    fps = float(batch_item(batch, 'fps', default=float('nan')))
    window_span_seconds = float(batch_item(batch, 'window_span_seconds', default=float('nan')))
    frames_np = batch['sequence'][0, :, 0].detach().cpu().numpy().copy()
    dense_masks, mask_is_gt = pseudolabel_store.stack_for_frames(video_id, original_frame_indices)

    heatmaps = []
    n_layer_calls = 0
    center_idx = SEQUENCE_LENGTH // 2
    pred_mask = np.zeros_like(dense_masks[center_idx], dtype=np.float32)
    dice = float('nan')
    pred_iou = float('nan')
    for sequence_idx, target_frame_idx in enumerate(original_frame_indices):
        shifted_sequence, _shifted_frame_indices = read_temporal_sequence(video_id, target_frame_idx, stride)
        shifted_sequence = shifted_sequence.to(device)
        target_mask_np, _is_gt = pseudolabel_store.get(video_id, target_frame_idx)
        target_mask = torch.from_numpy(target_mask_np).unsqueeze(0).unsqueeze(0).to(device)
        try:
            result = convlstm_gradcam(model, shifted_sequence, target_mask, 'decoder3')
            heatmaps.append(np.asarray(result.heatmaps, dtype=np.float32)[SEQUENCE_LENGTH // 2])
            if sequence_idx == center_idx:
                pred_mask, dice, pred_iou = prediction_mask_and_scores(result.logits, dense_masks[center_idx])
            n_layer_calls += int(result.n_layer_calls)
        finally:
            del shifted_sequence, target_mask
            cleanup_gradcam_gpu(model)

    heatmaps = np.stack(heatmaps, axis=0).astype(np.float32)
    metadata = {
        'sample_id': sample_id,
        'video_id': video_id,
        'center_frame_idx': center_frame_idx,
        'frame_indices': ' '.join(str(x) for x in original_frame_indices),
        'fps': fps,
        'window_span_seconds': window_span_seconds,
        'model_family': 'convlstm_unet',
        'model_id': model_id,
        'temporal_stride': stride,
        'target_layer': display_layer,
        'checkpoint_path': str(checkpoint_path),
        'n_layer_calls_captured': n_layer_calls,
        'gradcam_protocol': 'sliding_center_frame_for_each_sequence_position',
        'mask_sources': ' '.join('gt' if x else 'pseudo' for x in mask_is_gt),
        'center_dice': dice,
        'center_pred_iou': pred_iou,
    }
    metric_row = compute_temporal_saliency_metrics(heatmaps, dense_masks, metadata=metadata, saliency_percentile=SALIENCY_PERCENTILE)
    per_sample_rows.append(metric_row)
    record_frame_overlaps(metric_row, model_id, display_layer, original_frame_indices, mask_is_gt)
    cam_path = write_cam_artifact(heatmaps, model_id, display_layer, sample_id, metadata)
    cam_manifest_rows.append({**metadata, 'cam_npz_path': cam_path})
    maybe_store_qualitative(display_layer, sample_id, model_id, frames_np, heatmaps, dense_masks, original_frame_indices, metric_row, pred_mask, dice, pred_iou)


def evaluate_convlstm_stride(stride: int):
    checkpoint_path = convlstm_checkpoint_for_stride(stride)
    if checkpoint_path is None or not checkpoint_path.exists():
        warnings.warn(f'Skipping ConvLSTM stride {stride}: checkpoint missing at {checkpoint_path}')
        return

    model, config = build_convlstm_for_stride(stride)
    load_state_dict_from_checkpoint(model, checkpoint_path)
    loader = make_temporal_loader(stride)
    model_id = f'convlstm_stride_{stride}'

    try:
        for batch in tqdm(loader, desc=model_id, leave=False):
            sequence = batch['sequence'].to(device)
            center_mask = batch['mask'].to(device)
            try:
                sample_id = sanitize_name(batch_item(batch, 'id'))
                video_id = batch_item(batch, 'video_id')
                center_frame_idx = int(batch_item(batch, 'frame_idx'))
                frame_indices = [int(x) for x in batch_item(batch, 'frame_indices')]
                fps = float(batch_item(batch, 'fps', default=float('nan')))
                window_span_seconds = float(batch_item(batch, 'window_span_seconds', default=float('nan')))
                frames_np = sequence[0, :, 0].detach().cpu().numpy().copy()
                dense_masks, mask_is_gt = pseudolabel_store.stack_for_frames(video_id, frame_indices)

                convlstm_layers = [
                    ('convlstm_bottleneck_encoder', 'bottleneck_encoder'),
                    ('convlstm_temporal_bottleneck', 'temporal_bottleneck'),
                ]
                for display_layer, layer_name in convlstm_layers:
                    result = None
                    heatmaps = None
                    try:
                        result = convlstm_gradcam(model, sequence, center_mask, layer_name)
                        heatmaps = np.asarray(result.heatmaps, dtype=np.float32).copy()
                        n_layer_calls = result.n_layer_calls
                        center_idx = SEQUENCE_LENGTH // 2
                        pred_mask, dice, pred_iou = prediction_mask_and_scores(result.logits, dense_masks[center_idx])
                        metadata = {
                            'sample_id': sample_id,
                            'video_id': video_id,
                            'center_frame_idx': center_frame_idx,
                            'frame_indices': ' '.join(str(x) for x in frame_indices),
                            'fps': fps,
                            'window_span_seconds': window_span_seconds,
                            'model_family': 'convlstm_unet',
                            'model_id': model_id,
                            'temporal_stride': stride,
                            'target_layer': display_layer,
                            'raw_target_layer': layer_name,
                            'checkpoint_path': str(checkpoint_path),
                            'n_layer_calls_captured': n_layer_calls,
                            'gradcam_protocol': 'single_inference_existing_protocol',
                            'mask_sources': ' '.join('gt' if x else 'pseudo' for x in mask_is_gt),
                            'center_dice': dice,
                            'center_pred_iou': pred_iou,
                        }
                        metric_row = compute_temporal_saliency_metrics(heatmaps, dense_masks, metadata=metadata, saliency_percentile=SALIENCY_PERCENTILE)
                        per_sample_rows.append(metric_row)
                        record_frame_overlaps(metric_row, model_id, display_layer, frame_indices, mask_is_gt)
                        cam_path = write_cam_artifact(heatmaps, model_id, display_layer, sample_id, metadata)
                        cam_manifest_rows.append({**metadata, 'cam_npz_path': cam_path})
                        maybe_store_qualitative(display_layer, sample_id, model_id, frames_np, heatmaps, dense_masks, frame_indices, metric_row, pred_mask, dice, pred_iou)
                    finally:
                        del result, heatmaps
                        cleanup_gradcam_gpu(model)

                evaluate_convlstm_decoder3_sliding(model, stride, batch, checkpoint_path, model_id)
            finally:
                del sequence, center_mask
                cleanup_gradcam_gpu(model)
    finally:
        del loader, model
        cleanup_gradcam_gpu()


def evaluate_unet_on_stride_sequences(stride: int, model, layer_name: str, checkpoint_path: Path, layer_label: str):
    loader = make_temporal_loader(stride)
    model_id = f'unet2d_on_stride_{stride}_frames'
    try:
        for batch in tqdm(loader, desc=f'{model_id}_{sanitize_name(layer_label)}', leave=False):
            sequence = batch['sequence'].to(device)
            try:
                sample_id = sanitize_name(batch_item(batch, 'id'))
                video_id = batch_item(batch, 'video_id')
                center_frame_idx = int(batch_item(batch, 'frame_idx'))
                frame_indices = [int(x) for x in batch_item(batch, 'frame_indices')]
                fps = float(batch_item(batch, 'fps', default=float('nan')))
                window_span_seconds = float(batch_item(batch, 'window_span_seconds', default=float('nan')))
                frames_np = sequence[0, :, 0].detach().cpu().numpy().copy()
                dense_masks, mask_is_gt = pseudolabel_store.stack_for_frames(video_id, frame_indices)
                dense_masks_t = torch.from_numpy(dense_masks).unsqueeze(0).unsqueeze(2).to(device)

                result = None
                heatmaps = None
                try:
                    result = unet_framewise_gradcam(model, sequence, dense_masks_t, layer_name)
                    heatmaps = np.asarray(result.heatmaps, dtype=np.float32).copy()
                    n_layer_calls = result.n_layer_calls
                    center_idx = SEQUENCE_LENGTH // 2
                    pred_mask, dice, pred_iou = prediction_mask_and_scores(result.logits[0, center_idx, 0], dense_masks[center_idx])
                    metadata = {
                        'sample_id': sample_id,
                        'video_id': video_id,
                        'center_frame_idx': center_frame_idx,
                        'frame_indices': ' '.join(str(x) for x in frame_indices),
                        'fps': fps,
                        'window_span_seconds': window_span_seconds,
                        'model_family': 'unet_2d',
                        'model_id': model_id,
                        'temporal_stride': stride,
                        'target_layer': layer_label,
                        'raw_target_layer': layer_name,
                        'checkpoint_path': str(checkpoint_path),
                        'n_layer_calls_captured': n_layer_calls,
                        'gradcam_protocol': 'framewise_unet_frame_specific_mask_targets',
                        'mask_sources': ' '.join('gt' if x else 'pseudo' for x in mask_is_gt),
                        'center_dice': dice,
                        'center_pred_iou': pred_iou,
                    }
                    metric_row = compute_temporal_saliency_metrics(heatmaps, dense_masks, metadata=metadata, saliency_percentile=SALIENCY_PERCENTILE)
                    per_sample_rows.append(metric_row)
                    record_frame_overlaps(metric_row, model_id, layer_label, frame_indices, mask_is_gt)
                    cam_path = write_cam_artifact(heatmaps, model_id, layer_label, sample_id, metadata)
                    cam_manifest_rows.append({**metadata, 'cam_npz_path': cam_path})
                    maybe_store_qualitative(layer_label, sample_id, model_id, frames_np, heatmaps, dense_masks, frame_indices, metric_row, pred_mask, dice, pred_iou)
                finally:
                    del result, heatmaps, dense_masks_t
                    cleanup_gradcam_gpu(model)
            finally:
                del sequence
                cleanup_gradcam_gpu(model)
    finally:
        del loader
        cleanup_gradcam_gpu(model)


def save_layer_comparison_figures() -> pd.DataFrame:
    rows = []
    for layer_key, samples_by_id in qualitative_by_layer.items():
        layer_dir = QUAL_DIR / layer_key
        layer_dir.mkdir(parents=True, exist_ok=True)
        for sample_id, models_for_sample in samples_by_id.items():
            model_items = sorted(models_for_sample.items(), key=lambda item: item[0])
            if not model_items:
                continue
            n_overlay_cols = SEQUENCE_LENGTH
            n_cols = n_overlay_cols + 2
            fig, axes = plt.subplots(
                len(model_items),
                n_cols,
                figsize=(3.0 * n_cols, 2.9 * len(model_items)),
                squeeze=False,
            )
            center_idx = SEQUENCE_LENGTH // 2
            for row_idx, (model_id, payload) in enumerate(model_items):
                frames = payload['frames']
                cams = payload['heatmaps']
                pred_mask = payload['pred_mask']
                frame_indices = payload['frame_indices']
                metric_row = payload['metric_row']
                dice = payload.get('dice', metric_row.get('center_dice', np.nan))
                pred_iou = payload.get('pred_iou', metric_row.get('center_pred_iou', np.nan))
                mean_overlap = metric_row.get('dense_saliency_mask_overlap_mean', np.nan)

                label_ax = axes[row_idx, 0]
                label_ax.axis('off')
                label_ax.text(
                    0.5,
                    0.5,
                    f"{model_id}\n{payload.get('target_layer', metric_row.get('target_layer', layer_key))}\nDice={dice:.3f}\nPred IoU={pred_iou:.3f}\nCAM/LV overlap={mean_overlap:.3f}",
                    ha='center',
                    va='center',
                    fontsize=8,
                    wrap=True,
                )

                mask_ax = axes[row_idx, 1]
                mask_ax.imshow(pred_mask, cmap='gray', vmin=0, vmax=1)
                mask_ax.set_title('pred center mask', fontsize=8)
                mask_ax.axis('off')

                for time_idx in range(n_overlay_cols):
                    ax = axes[row_idx, time_idx + 2]
                    ax.imshow(frames[time_idx], cmap='gray', vmin=0, vmax=1)
                    ax.imshow(cams[time_idx], cmap='magma', vmin=0, vmax=1, alpha=0.45)
                    target_tag = ' target' if time_idx == center_idx else ''
                    frame_idx = frame_indices[time_idx] if time_idx < len(frame_indices) else time_idx
                    ax.set_title(f't{time_idx} frm {frame_idx}{target_tag}', fontsize=8)
                    ax.axis('off')
            fig.suptitle(f'{sample_id} | {layer_key} | predicted mask + full 5-frame CAM overlays', fontsize=12)
            fig.tight_layout(rect=(0, 0, 1, 0.94))
            out_path = layer_dir / f'{sample_id}_{layer_key}_predmask_full_sequence_model_comparison.png'
            fig.savefig(out_path, dpi=150, bbox_inches='tight')
            plt.close(fig)
            rows.append({'sample_id': sample_id, 'layer': layer_key, 'figure_path': str(out_path)})
    return pd.DataFrame(rows)


In [ ]:
for stride in CONVLSTM_STRIDES:
    evaluate_convlstm_stride(stride)

if UNET_CHECKPOINT_PATH is None or not UNET_CHECKPOINT_PATH.exists():
    warnings.warn(f'Skipping 2D U-Net baseline: checkpoint missing at {UNET_CHECKPOINT_PATH}')
else:
    unet_model = build_unet_baseline()
    load_state_dict_from_checkpoint(unet_model, UNET_CHECKPOINT_PATH)
    unet_final_layer = find_last_conv2d_name(unet_model)
    unet_bottleneck_layer = find_unet_encoder_bottleneck_conv2d_name(unet_model)
    unet_layers = [
        ('unet_final_convolution', unet_final_layer),
        ('unet_encoder_bottleneck', unet_bottleneck_layer),
    ]
    print(f'2D U-Net final convolution Grad-CAM layer: {unet_final_layer}')
    print(f'2D U-Net encoder bottleneck Grad-CAM layer: {unet_bottleneck_layer}')
    for stride in CONVLSTM_STRIDES:
        for layer_label, layer_name in unet_layers:
            evaluate_unet_on_stride_sequences(stride, unet_model, layer_name, UNET_CHECKPOINT_PATH, layer_label)

per_sample_df = pd.DataFrame(per_sample_rows)
frame_overlap_df = pd.DataFrame(frame_overlap_rows)
cam_manifest_df = pd.DataFrame(cam_manifest_rows)
per_sample_df.to_csv(METRICS_DIR / 'per_sample_metrics.csv', index=False)
per_sample_df.to_csv(TABLES_DIR / 'per_sample_metrics.csv', index=False)
frame_overlap_df.to_csv(METRICS_DIR / 'framewise_saliency_mask_overlap.csv', index=False)
cam_manifest_df.to_csv(TABLES_DIR / 'cam_manifest.csv', index=False)
qualitative_df = save_layer_comparison_figures()
qualitative_df.to_csv(TABLES_DIR / 'qualitative_visualization_manifest.csv', index=False)
print(f'Per-sample rows: {len(per_sample_df):,}')
print(f'Frame-overlap rows: {len(frame_overlap_df):,}')
print(f'CAM artifacts: {len(cam_manifest_df):,}')
print(f'Qualitative figures: {len(qualitative_df):,}')
per_sample_df.head()


## Aggregate comparisons and plots

In [ ]:
aggregated_df = aggregate_metrics(per_sample_df)
aggregated_df.to_csv(METRICS_DIR / 'aggregated_metrics.csv', index=False)
aggregated_df.to_csv(TABLES_DIR / 'aggregated_metrics.csv', index=False)

if not per_sample_df.empty:
    stride_comparison = aggregated_df[aggregated_df['model_family'] == 'convlstm_unet'].copy()
    layer_comparison = aggregated_df.copy()
    frame_overlap_df.to_csv(TABLES_DIR / 'framewise_saliency_mask_overlap.csv', index=False)
    model_comparison = aggregated_df.groupby(['model_family', 'target_layer'], dropna=False).mean(numeric_only=True).reset_index()
else:
    stride_comparison = pd.DataFrame()
    layer_comparison = pd.DataFrame()
    model_comparison = pd.DataFrame()

stride_comparison.to_csv(TABLES_DIR / 'comparison_across_strides.csv', index=False)
layer_comparison.to_csv(TABLES_DIR / 'comparison_across_target_layers.csv', index=False)
model_comparison.to_csv(TABLES_DIR / 'comparison_convlstm_vs_unet.csv', index=False)
save_metric_plots(aggregated_df, FIGURES_DIR)

summary = {
    'run_mode': RUN_MODE,
    'official_test_samples_available': len(test_samples),
    'evaluation_samples': len(eval_samples),
    'convlstm_target_layers': ['convlstm_bottleneck_encoder', 'convlstm_temporal_bottleneck', 'convlstm_decoder3_sliding_window'],
    'convlstm_strides': CONVLSTM_STRIDES,
    'saliency_percentile': SALIENCY_PERCENTILE,
    'max_overlay_samples': MAX_OVERLAY_SAMPLES,
    'overlay_samples_saved': len(overlay_sample_ids),
    'notes': 'Dense frame-wise LV masks are loaded from notebook 18 pseudolabel outputs. Ground-truth ED/ES masks are used when available; otherwise pseudolabel masks are used. Existing temporal metrics are unchanged except saliency-mask overlap is now dense frame-wise.',
    'pseudolabel_run_dir': str(PSEUDOLABEL_RUN_DIR),
    'cam_storage': 'compressed uint8 NPZ files under cams/',
}
with (OUTPUT_DIR / 'gradcam_temporal_evaluation_summary.json').open('w', encoding='utf-8') as file:
    json.dump(summary, file, indent=2)

aggregated_df


## Output checks

In [ ]:
required_outputs = [
    METRICS_DIR / 'per_sample_metrics.csv',
    METRICS_DIR / 'aggregated_metrics.csv',
    TABLES_DIR / 'comparison_across_strides.csv',
    TABLES_DIR / 'comparison_across_target_layers.csv',
    TABLES_DIR / 'comparison_convlstm_vs_unet.csv',
    TABLES_DIR / 'checkpoint_discovery.csv',
    TABLES_DIR / 'cam_manifest.csv',
    TABLES_DIR / 'framewise_saliency_mask_overlap.csv',
    TABLES_DIR / 'qualitative_visualization_manifest.csv',
    OUTPUT_DIR / 'gradcam_temporal_evaluation_summary.json',
]
missing = [path for path in required_outputs if not path.exists()]
assert not missing, f'Missing expected outputs: {missing}'
print(f'Grad-CAM temporal evaluation outputs saved to: {OUTPUT_DIR.resolve()}')
